In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
crm_customers = spark.read.format("delta").table("data_engineering_2026.silver.silver_crm_customers")
erp_customers = spark.read.format("delta").table("data_engineering_2026.silver.silver_erp_customers")
erp_locations = spark.read.format("delta").table("data_engineering_2026.silver.silver_erp_locations")

In [0]:
gold_customers = crm_customers.alias("crm")\
    .join(erp_customers.alias("erp"),col("crm.customer_key") == col("erp.customer_key"), "left")\
    .join(erp_locations.alias("loc"),col("crm.customer_key") == col("loc.customer_key"), "left") \
    .select(
        col("crm.customer_id"),
        col("crm.customer_key"),
        col("crm.full_name"),
        col("crm.first_name"),
        col("crm.last_name"),
        col("crm.martial_Status"),
        col("crm.gender").alias("gender"),
        col("crm.create_date"),
        col("erp.birth_date"),
        col("loc.country")
    )

In [0]:
gold_customers.display()

In [0]:

gold_customers.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in gold_customers.columns
]).show()

In [0]:
gold_customers.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("data_engineering_2026.gold.gold_customers")

## Gold_Products

In [0]:
crm_products = spark.read.format("delta").table("data_engineering_2026.silver.silver_crm_products")
erp_categories = spark.read.format("delta").table("data_engineering_2026.silver.silver_erp_categories")

In [0]:
gold_products = crm_products.alias("crm")\
    .join(erp_categories.alias("erp"),col("crm.category_id") == col("erp.category_id"), "left")


In [0]:
windows = Window.partitionBy("product_number").orderBy(col("start_date").desc())

gold_products = gold_products.withColumn(
    "rn", row_number().over(windows))\
    .filter(col("rn") == 1)\
    .drop("rn")

In [0]:
gold_products = gold_products.select(
    col("crm.product_id"),
    col("crm.product_number"),
    col("crm.product_name"),
    col("crm.product_cost"),
    col("crm.product_line"),
    col("crm.start_date"),
    col("crm.end_date"),
    col("crm.category_id"),
    col("erp.category"),
    col("erp.sub_category"),
    col("erp.maintenance_flag")
)


In [0]:
gold_products\
    .display()  

In [0]:
gold_products.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in gold_products.columns
]).show()

In [0]:
gold_products.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable("data_engineering_2026.gold.gold_products")

## Gold_Sales

In [0]:
crm_sales = spark.read.format("delta").table("data_engineering_2026.silver.silver_crm_sales")
erp_locations = spark.read.format("delta").table("data_engineering_2026.silver.silver_erp_locations")
gold_customers = spark.read.format("delta").table("data_engineering_2026.gold.gold_customers")
gold_products = spark.read.format("delta").table("data_engineering_2026.gold.gold_products")




In [0]:
gold_sales = crm_sales.alias("crm")\
	.join(gold_customers.alias("gold_cus"), col("crm.customer_id") == col("gold_cus.customer_id"), "left")\
	.join(gold_products.alias("gold_prod"), col("crm.product_number") == col("gold_prod.product_number"), "left")


In [0]:
gold_sales = gold_sales.select(
	col("crm.order_number"),
	col("crm.order_date"),
	col("crm.ship_date"),
	col("crm.due_date"),
	col("gold_cus.customer_key"),
 	col("gold_prod.product_name"),
	col("gold_cus.full_name"),
 	col("gold_cus.country"),
	col("crm.product_number"),
	col("crm.sales_amount"),
	col("crm.quantity"),
	col("crm.price")
)

gold_sales.display()


In [0]:
gold_sales.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .saveAsTable("data_engineering_2026.gold.gold_sales")